In [ ]:
##modules
#%matplotlib widget
#%matplotlib inline
#
# %matplotlib qt
import mne
import numpy as np

# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.
import matplotlib
import matplotlib.pyplot as plt

matplotlib.use('Qt5Agg')  # Asegúrate de que este backend está instalado.
mne.viz.set_browser_backend('qt')  # o 'matplotlib'

import pandas as pd 
import os
import sys

import re
from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd
sys.path.append("..")  # esto sube un nivel desde Scripts_visual_block

from scipy.io import savemat


In [ ]:

# --- Configuración dinámica de rutas ---
from get_paths_SELF_local import get_paths_SELF

# Parámetros editables
disco = "c"
layer_script = "event"
subj = "s01b"

# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


In [ ]:
dict_annotations = {
    'self_pos': self_pos,
    'self_neu': self_neu,
    'self_neg': self_neg,
    'friend_pos': friend_pos,
    'friend_neu': friend_neu,
    'friend_neg': friend_neg,
    'unk_pos': unk_pos,
    'unk_neu': unk_neu,
    'unk_neg': unk_neg
}


def export_epochs_to_fieldtrip(epochs, filename):
    """
    Convierte un objeto mne.Epochs a formato FieldTrip y lo guarda como .mat

    Parámetros
    ----------
    epochs : mne.Epochs
        Objeto de MNE con datos epocados.
    filename : str
        Nombre del archivo .mat a guardar (incluye extensión).
    """
    # Crear estructura FieldTrip
    ft_data = {}
    
    # FieldTrip espera una lista de matrices (nchan x ntime) por trial
    ft_data['trial'] = [trial for trial in epochs.get_data().transpose(0, 2, 1)]
    
    # Vector de tiempos repetido para cada trial
    times = epochs.times.tolist()
    ft_data['time'] = [times for _ in range(len(epochs))]
    
    # Nombres de canales
    ft_data['label'] = epochs.ch_names
    
    # Frecuencia de muestreo
    ft_data['fsample'] = float(epochs.info['sfreq'])
    
    # Guardar como struct en .mat
    savemat(filename, {'data': ft_data})
    print(f"✅ Exportado {filename} en formato FieldTrip.")
